# Baseline Model Training for Migraine Occurrence Prediction

This notebook trains and compares two baseline machine-learning models for predicting `migraine_occurrence`. The purpose is to establish an initial performance benchmark before any advanced modelling or optimisation is attempted.

## 1. Import required libraries

The following libraries are used for data handling, data splitting, feature scaling, model training, and model evaluation.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

## 2. Load the dataset

The dataset is loaded using a path-checking approach so the notebook can still run if the kernel starts from either the project root or the `notebooks` folder. The code below prints the exact file path that is used.

In [ ]:
candidate_paths = [
    Path('data/migraine_dataset.csv'),
    Path('../data/migraine_dataset.csv'),
    Path.cwd() / 'data' / 'migraine_dataset.csv',
    Path.cwd().parent / 'data' / 'migraine_dataset.csv'
]

data_path = next((path for path in candidate_paths if path.exists()), None)

if data_path is None:
    raise FileNotFoundError('Could not find migraine_dataset.csv in the expected data folder.')

print('Loading dataset from:', data_path)
df = pd.read_csv(data_path)
df.head()

## 3. Define `X` and `y`

The input features are the selected lifestyle variables, and the target variable is `migraine_occurrence`.

In [ ]:
features = [
    'sleep_hours',
    'mood_level',
    'stress_level',
    'hydration_level',
    'screen_time'
]

target = 'migraine_occurrence'

X = df[features]
y = df[target]

print('Shape of X:', X.shape)
print('Shape of y:', y.shape)

## 4. Split the data into training and testing sets

The dataset is divided into training and testing subsets. Stratification is used so that the class distribution remains similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print('X_train shape:', X_train.shape)
print('X_test shape:', X_test.shape)
print('y_train shape:', y_train.shape)
print('y_test shape:', y_test.shape)

## 5. Scale the input features for Logistic Regression

Logistic Regression often performs better when numerical input features are standardised. The scaler is fitted on the training data and then applied to both the training and testing sets.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

## 6. Train a Logistic Regression model

Logistic Regression is used as the first baseline model because it is a common and interpretable method for binary classification.

In [ ]:
log_reg = LogisticRegression(random_state=42, max_iter=1000)
log_reg.fit(X_train_scaled, y_train)

## 7. Evaluate Logistic Regression

The model is evaluated using several standard binary classification metrics. These measures help us understand both overall accuracy and the model's ability to detect positive migraine cases.

In [ ]:
y_pred_lr = log_reg.predict(X_test_scaled)
y_prob_lr = log_reg.predict_proba(X_test_scaled)[:, 1]

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_f1 = f1_score(y_test, y_pred_lr)
lr_roc_auc = roc_auc_score(y_test, y_prob_lr)

print('Logistic Regression Performance')
print('Accuracy:', lr_accuracy)
print('Precision:', lr_precision)
print('Recall:', lr_recall)
print('F1-score:', lr_f1)
print('ROC-AUC:', lr_roc_auc)
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_lr))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_lr))

## 8. Train a Random Forest Classifier

Random Forest is used as a second baseline model because it can capture non-linear relationships and interactions between variables. Class balancing is applied to support fairer learning across the target classes.

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight='balanced'
)
rf_model.fit(X_train, y_train)

## 9. Evaluate Random Forest

The Random Forest model is evaluated using the same set of metrics so that a direct comparison can be made.

In [ ]:
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_precision = precision_score(y_test, y_pred_rf)
rf_recall = recall_score(y_test, y_pred_rf)
rf_f1 = f1_score(y_test, y_pred_rf)
rf_roc_auc = roc_auc_score(y_test, y_prob_rf)

print('Random Forest Performance')
print('Accuracy:', rf_accuracy)
print('Precision:', rf_precision)
print('Recall:', rf_recall)
print('F1-score:', rf_f1)
print('ROC-AUC:', rf_roc_auc)
print('\nConfusion Matrix:')
print(confusion_matrix(y_test, y_pred_rf))
print('\nClassification Report:')
print(classification_report(y_test, y_pred_rf))

## 10. Create a comparison table

A simple summary table is created to compare the performance of both baseline models across the main evaluation metrics.

In [ ]:
comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest'],
    'Accuracy': [lr_accuracy, rf_accuracy],
    'Precision': [lr_precision, rf_precision],
    'Recall': [lr_recall, rf_recall],
    'F1-score': [lr_f1, rf_f1],
    'ROC-AUC': [lr_roc_auc, rf_roc_auc]
})

comparison_df

## 11. Conclusion

The comparison table above should be used to identify the stronger baseline model. In general, the better model is the one with the higher F1-score and ROC-AUC, because these measures provide a balanced view of classification performance and discrimination ability.

After running the notebook, this section can be updated in the project report to state which of the two baseline models performed better.